在仿真过程中，如果您希望某个参数（比如控制器增益、扰动强度、风速等）在不同的时间段取不同的值，可以通过以下几种常见的方法在 Python 代码中实现：

方法一：在主仿真循环中使用 if/elif/else 语句

这是最直接和常用的方法。在 simulation.py 的主循环内部，根据当前的仿真时间 t 来判断应该使用哪组参数值。

2.1 控制目标
对于轨迹跟踪问题，我们的控制目标是：

- 最小化实际轨迹与期望轨迹之间的误差
- 最小化控制输入的能量消耗
- 满足系统约束条件

In [ ]:
# 代价函数设计
J = sum(w_e * ||e1||^2 + w_v * ||e2||^2 + w_u * ||u||^2)

其中：

- e1 = y - yc (位置和姿态误差)
- e2 = y_dot - yc_dot (速度误差)
- u = tau (控制输入)
- w_e, w_v, w_u 为权重系数

In [ ]:
# simulation.py (部分 - 主循环内)
import numpy as np
import parameters as params # 假设参数初始值定义在这里
# ... 其他导入 ...

# --- 仿真循环 ---
# ... (初始化等代码) ...

for i, t in enumerate(sim_time):
    # --- 1. 根据时间调整参数 (Adjust parameters based on time) ---
    current_k1 = params.k1 # 默认使用 parameters.py 中的值
    current_disturbance_scale = 5000 # 默认扰动尺度
    current_wind = params.V_WIND_ERF # 默认风速

    if t < 50.0:
        # 时间段 1: 0 <= t < 50 秒
        # 可以保持默认值，或者设置特定的值
        print(f"时间 {t:.2f}s: 使用第一阶段参数")
        # current_k1 = np.array([0.05, 0.06, 0.05, 0.1, 0.1, 0.2]) # 示例：降低增益
    elif t < 120.0:
        # 时间段 2: 50 <= t < 120 秒
        print(f"时间 {t:.2f}s: 使用第二阶段参数")
        current_k1 = np.array([0.1, 0.12, 0.1, 0.3, 0.3, 0.5]) # 示例：提高增益
        current_disturbance_scale = 7000 # 示例：增大扰动
        current_wind = np.array([8.0, 3.0, 0.0]) # 示例：改变风速
    else:
        # 时间段 3: t >= 120 秒
        print(f"时间 {t:.2f}s: 使用第三阶段参数（可能恢复默认或使用新值）")
        # 可以恢复默认值
        current_k1 = params.k1
        current_disturbance_scale = 5000
        current_wind = params.V_WIND_ERF
        # 或者设置第三阶段的值
        # current_k1 = np.array([...])

    # --- 2. 在后续计算中使用调整后的参数 ---
    # 获取当前状态、期望状态等...

    # 更新扰动观测器 (如果需要调整观测器参数 l1-l5, beta1/2)
    # observer.l1 = new_l1_value # 如果需要修改
    # delta_hat = observer.update(...)

    # 计算控制输入 (使用调整后的增益 current_k1 等)
    # 确保 controller.calculate_control 使用的是 current_k1, k3, k4
    # 如果控制器内部直接引用 params.k1，需要修改控制器类允许传入增益
    # 或者在控制器类内部也加入时间判断
    # 假设 controller.calculate_control 可以接收增益作为参数 (推荐):
    # tau = controller.calculate_control(t, e1, e2, delta_hat, gamma, gamma_d, xc, xc_dot,
    #                                   k1=current_k1, k3=params.k3, k4=params.k4) # 传递调整后的k1

    # 获取实际扰动 (使用调整后的尺度)
    actual_delta = params.disturbance_delta(t, scale=current_disturbance_scale) # 需要修改 disturbance_delta 函数接受 scale 参数

    # 积分气艇模型 (如果风速变化，需要传递调整后的风速)
    def airship_ode(t_rk, X_rk):
        # ...
        # 获取当时的控制 tau (可能也基于调整后的增益)
        # 获取当时的扰动 d (可能基于调整后的尺度)
        current_delta_rk = params.disturbance_delta(t_rk, scale=current_disturbance_scale)
        # 获取当时的风速 wind_rk (可能基于调整后的风速)
        current_wind_rk = current_wind # 假设风速在小步长内不变
        # 修改 airship.rhs 让其能接收并使用变化的风速
        return airship.rhs(t_rk, X_rk, tau, lambda time_ignored: current_delta_rk, wind_erf=current_wind_rk) # 假设 rhs 能处理风速

    # ... (RK4 积分) ...

    # ... (记录数据) ...

实现此方法需要注意：
参数传递: 确保那些需要根据时间调整的参数（如 current_k1, current_disturbance_scale, current_wind）被正确地传递给使用它们的方法（如 controller.calculate_control, params.disturbance_delta, airship.rhs）。这可能需要修改这些方法的签名（增加参数）。
修改函数/方法:

可能需要修改 params.disturbance_delta 函数，让它接受一个可选的 scale 参数。

可能需要修改 airship.rhs 方法，让它接受一个 wind_erf 参数，并在计算相对速度时使用它，而不是直接从 self 或 params 读取。

可能需要修改 controller.calculate_control 方法，让它接受 k1, k3, k4 等增益作为参数，而不是在内部硬编码或直接读取 params。

方法二：定义时间分段函数

对于更复杂的参数变化逻辑，或者为了让主循环更简洁，可以定义一个函数来根据时间返回相应的参数值。

In [ ]:
# parameters.py (或一个单独的 parameter_schedule.py)

import numpy as np

def get_scheduled_params(t):
    """根据时间返回参数字典"""
    if t < 50.0:
        k1 = np.array([0.07, 0.08, 0.07, 0.2, 0.2, 0.4])
        disturbance_scale = 5000
        wind_erf = np.array([5.0, 2.0, 0.0])
    elif t < 120.0:
        k1 = np.array([0.1, 0.12, 0.1, 0.3, 0.3, 0.5])
        disturbance_scale = 7000
        wind_erf = np.array([8.0, 3.0, 0.0])
    else:
        k1 = np.array([0.07, 0.08, 0.07, 0.2, 0.2, 0.4]) # 恢复默认
        disturbance_scale = 5000
        wind_erf = np.array([5.0, 2.0, 0.0])

    # 返回包含所有时变参数的字典
    return {
        "k1": k1,
        "disturbance_scale": disturbance_scale,
        "wind_erf": wind_erf
        # 可以添加其他需要调度的参数
    }

# 可能还需要修改扰动函数以接受 scale
def disturbance_delta(t, scale=5000):
    """定义外部扰动向量，带有可调尺度"""
    d_vec = np.zeros(6)
    d_vec[0] = 0.5 + 2 * np.sin(0.1 * t)
    # ... (其他分量) ...
    d_vec[5] = 1.5 + 2 * np.cos(0.1 * t)
    return scale * d_vec

In [ ]:
# simulation.py (部分 - 主循环内)
# from parameter_schedule import get_scheduled_params # 如果单独存放
from parameters import get_scheduled_params, disturbance_delta # 假设放在 parameters.py

# ...

for i, t in enumerate(sim_time):
    # --- 1. 获取当前时间点的参数 ---
    scheduled_params = get_scheduled_params(t)
    current_k1 = scheduled_params["k1"]
    current_disturbance_scale = scheduled_params["disturbance_scale"]
    current_wind = scheduled_params["wind_erf"]

    # --- 2. 在后续计算中使用获取的参数 ---
    # ... (与方法一类似，确保参数正确传递) ...

    # 计算控制输入
    # tau = controller.calculate_control(..., k1=current_k1, ...)

    # 获取实际扰动
    actual_delta = disturbance_delta(t, scale=current_disturbance_scale)

    # 积分气艇模型
    def airship_ode(t_rk, X_rk):
        # ...
        current_delta_rk = disturbance_delta(t_rk, scale=current_disturbance_scale)
        current_wind_rk = current_wind
        return airship.rhs(t_rk, X_rk, tau, lambda time_ignored: current_delta_rk, wind_erf=current_wind_rk)
    # ... (RK4) ...

方法三：使用插值函数（例如 scipy.interpolate.interp1d）

如果参数需要在时间点之间平滑过渡，可以使用插值。

In [ ]:
# parameters.py (或 parameter_schedule.py)
import numpy as np
from scipy.interpolate import interp1d

# 定义参数变化的时间点和对应的值
time_points = np.array([0.0,  49.9, 50.0, 119.9, 120.0, params.T_SPAN]) # 注意包含过渡点
k1_values_at_points = np.array([
    [0.07, 0.08, 0.07, 0.2, 0.2, 0.4], # t=0
    [0.07, 0.08, 0.07, 0.2, 0.2, 0.4], # t=49.9 (保持第一阶段)
    [0.1, 0.12, 0.1, 0.3, 0.3, 0.5],   # t=50.0 (切换到第二阶段)
    [0.1, 0.12, 0.1, 0.3, 0.3, 0.5],   # t=119.9 (保持第二阶段)
    [0.07, 0.08, 0.07, 0.2, 0.2, 0.4], # t=120.0 (切换回默认)
    [0.07, 0.08, 0.07, 0.2, 0.2, 0.4]  # t=T_SPAN (保持默认)
])
disturbance_scale_values = np.array([5000, 5000, 7000, 7000, 5000, 5000])
# ... 对其他参数也这样做 ...

# 创建插值函数 (kind='previous' 实现阶跃, kind='linear' 实现线性插值)
# 注意：需要对每个参数或每个参数的分量单独创建插值函数
k1_interp_funcs = [interp1d(time_points, k1_values_at_points[:, i], kind='previous', bounds_error=False, fill_value=(k1_values_at_points[0, i], k1_values_at_points[-1, i])) for i in range(6)]
disturbance_scale_interp = interp1d(time_points, disturbance_scale_values, kind='previous', bounds_error=False, fill_value=(disturbance_scale_values[0], disturbance_scale_values[-1]))
# ... 对风速等其他参数也创建插值函数 ...

def get_interpolated_params(t):
    """使用插值函数获取参数"""
    current_k1 = np.array([f(t) for f in k1_interp_funcs])
    current_disturbance_scale = disturbance_scale_interp(t)
    # ... 获取其他插值参数 ...
    current_wind = params.V_WIND_ERF # 示例：风速不插值
    return {
        "k1": current_k1,
        "disturbance_scale": current_disturbance_scale,
        "wind_erf": current_wind
    }

In [ ]:
# simulation.py (部分 - 主循环内)
from parameters import get_interpolated_params, disturbance_delta

# ...
for i, t in enumerate(sim_time):
    # --- 1. 获取当前时间点的插值参数 ---
    interpolated_params = get_interpolated_params(t)
    current_k1 = interpolated_params["k1"]
    current_disturbance_scale = interpolated_params["disturbance_scale"]
    current_wind = interpolated_params["wind_erf"]

    # --- 2. 在后续计算中使用获取的参数 ---
    # ... (与之前类似，确保参数传递) ...

选择哪种方法？
方法一 (if/elif/else)： 最简单直观，适用于只有少数几个时间段和少量参数变化的情况。

方法二 (时间分段函数)： 代码更整洁，逻辑更清晰，适用于参数变化较多或逻辑较复杂的情况。推荐使用这种方法来组织代码。

方法三 (插值函数)： 适用于需要参数在时间点之间平滑过渡的场景，或者当你有大量时间点数据需要处理时。实现起来稍微复杂一些。

根据您的具体需求选择最合适的方法。对于控制器增益或扰动强度的阶跃变化，方法一 或 方法二 通常足够。

在aero_coefficients.py中为什么要设计图中的这部分代码？ 我想学习这种写法

好的，if __name__ == "__main__": 这个代码块在 Python 脚本中是一个非常常见且有用的设计模式。我们来详细学习一下它的作用和为什么要在 aero_coefficients.py 中包含它：

if __name__ == "__main__": 的含义

__name__ 是什么?

__name__ 是 Python 中一个内置的特殊变量。

它的值取决于你如何运行这个 Python 脚本文件：

直接运行: 如果你直接在命令行中执行这个脚本（例如 python aero_coefficients.py），那么在这个脚本内部，__name__ 的值会被自动设置为字符串 "__main__"。

作为模块导入: 如果这个脚本被其他 Python 文件通过 import aero_coefficients 的方式导入并使用，那么在这个脚本内部，__name__ 的值会被设置为这个模块的名字，也就是字符串 "aero_coefficients"。

if __name__ == "__main__": 的作用:

这行代码检查当前脚本是被直接运行还是被作为模块导入。

只有当脚本被直接运行时，if 条件 (__name__ == "__main__") 才为真，其下面的代码块才会被执行。

如果脚本是被导入的，if 条件为假，其下面的代码块就不会被执行。

为什么要在 aero_coefficients.py 中设计这部分代码？

在 aero_coefficients.py 中加入 if __name__ == "__main__": 代码块的主要目的是：

提供测试功能 (Testing):

aero_coefficients.py 文件定义了重要的计算函数（calculate_added_mass_inertia_local 和 get_aero_coefficients）。我们希望能够独立地测试这些函数是否能正常工作，计算结果是否符合预期，而不需要运行整个复杂的仿真 (simulation.py)。

通过将测试代码放在 if __name__ == "__main__": 块内，我们可以直接运行 python aero_coefficients.py 来执行这些测试。测试代码会：

调用 calculate_added_mass_inertia_local() 计算 k1, k2。

调用 get_aero_coefficients() 计算所有的气动系数。

打印出计算结果，方便开发者检查。

捕获可能发生的 ValueError 等异常，并打印错误信息。

模块化和可重用性 (Modularity & Reusability):

这个文件的主要目的是提供计算气动系数的功能（通过函数）给其他模块（如 simulation.py）使用。

当 simulation.py 执行 import aero_coefficients 时，它只需要导入文件顶部的参数定义和函数定义，不需要执行测试代码。if __name__ == "__main__": 确保了这一点。

这使得 aero_coefficients.py 既可以作为一个可独立测试的单元，又可以作为一个干净的功能模块被其他代码导入和重用。

避免副作用 (Avoiding Side Effects):

如果测试代码（比如 print 语句或函数调用）没有放在 if __name__ == "__main__": 块内，那么每次其他文件 import aero_coefficients 时，这些测试代码都会被执行，这通常是我们不希望发生的“副作用”。我们只希望在导入时获得定义，而不是执行测试。

总结:

if __name__ == "__main__": 是一种将“脚本作为程序直接运行时要执行的代码”（通常是测试、示例用法或主程序逻辑）与“脚本作为模块导入时要提供的定义”（函数、类、常量）分离开的标准 Python 写法。

在 aero_coefficients.py 中使用它，可以让你方便地：

直接运行该文件来测试气动系数的计算是否正确。

放心地在 simulation.py 中 import aero_coefficients 来使用其提供的函数和参数，而不会意外地执行测试代码。

这是一种非常好的编程实践，有助于提高代码的可测试性、模块化和可维护性。

In [ ]:
# 记得去看 notion关于部分的截图笔记 
import numpy as np
import matplotlib.pyplot as plt
from airship.trajectory import define_spiral_trajectory

def trajectory_tracking():
    # Time range for simulation
    t_values = np.linspace(0, 200, 1000)  # Simulate for 200 seconds with 1000 points
    dt = t_values[1] - t_values[0]  # Time step

    # Initialize actual state
    pos_actual = np.array([0.0, 0.0, 0.0])  # Initial position
    vel_actual = np.array([0.0, 0.0, 0.0])  # Initial velocity

    # Initialize arrays to store results
    pos_actual_history = []
    pos_desired_history = []

    # Get the trajectory function
    trajectory_function = define_spiral_trajectory(t_values)

    # PD Controller gains
    Kp = np.array([1.0, 1.0, 1.0])  # Proportional gain
    Kd = np.array([0.5, 0.5, 0.5])  # Derivative gain

    # Simulation loop
    for t in t_values:
        # Get desired trajectory
        pos_desired, vel_desired, acc_desired = trajectory_function(t)

        # Compute errors
        e_pos = pos_actual - pos_desired
        e_vel = vel_actual - vel_desired

        # Compute control input (acceleration)
        u = -Kp * e_pos - Kd * e_vel

        # Update actual state
        pos_actual += vel_actual * dt
        vel_actual += u * dt

        # Store results
        pos_actual_history.append(pos_actual.copy())
        pos_desired_history.append(pos_desired.copy())

    # Convert results to arrays
    pos_actual_history = np.array(pos_actual_history)
    pos_desired_history = np.array(pos_desired_history)

    # Plot results
    plt.figure(figsize=(8, 6))
    plt.plot(pos_desired_history[:, 0], pos_desired_history[:, 1], label="Desired Trajectory", color="blue")
    plt.plot(pos_actual_history[:, 0], pos_actual_history[:, 1], label="Actual Trajectory", color="red", linestyle="--")
    plt.xlabel("X Position (m)")
    plt.ylabel("Y Position (m)")
    plt.title("Trajectory Tracking in X-Y Plane")
    plt.legend()
    plt.grid()
    plt.show()

# Run the trajectory tracking simulation
trajectory_tracking()

In [ ]:
# main_1.py

import argparse
import logging
import sys

from simulation.run_simulation import run_simulation

def parse_args():
    p = argparse.ArgumentParser(
        description="Airship Simulation"
    )
    p.add_argument(
        "-m", "--mode",
        choices=["debug", "release"],
        default="release",
        help="仿真模式：debug 会打印更多日志，release 只打印 INFO+"
    )
    p.add_argument(
        "-l", "--log-file",
        type=str,
        default=None,
        help="如果提供，日志也会写到这个文件"
    )
    p.add_argument(
        "-t", "--duration",
        type=float,
        default=None,
        help="可选：覆盖配置里的仿真总时长 T_SPAN（单位 s）"
    )
    return p.parse_args()

def setup_logger(mode: str, log_file: str = None):
    """
    根据模式和可选的文件路径，配置 root logger。
    debug 模式：DEBUG 级别；release 模式：INFO 级别
    """
    level = logging.DEBUG if mode == "debug" else logging.INFO

    # 创建 handler 列表：屏幕输出 + （可选）文件输出
    handlers = [logging.StreamHandler(sys.stdout)]
    if log_file:
        handlers.append(logging.FileHandler(log_file, encoding="utf-8"))

    logging.basicConfig(
        level=level,
        format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
        handlers=handlers
    )

    logger = logging.getLogger("main")
    logger.debug(f"Logger set to {mode.upper()} (level={level})")
    if log_file:
        logger.info(f"日志也写入：{log_file}")
    return logger

def main():
    args = parse_args()
    logger = setup_logger(args.mode, args.log_file)
    logger.info("程序启动")

    # 如果用户指定了 duration，就动态覆盖 config.parameters.T_SPAN
    if args.duration is not None:
        import config.parameters as params
        logger.info(f"覆盖仿真总时长：{params.T_SPAN} → {args.duration}")
        params.T_SPAN = args.duration

    # 启动仿真
    # run_simulation() 内部会使用 config.parameters 里的 DT、T_SPAN、X0 等
    run_simulation()

    logger.info("仿真结束")

if __name__ == "__main__":
    main()



'''
下面是一个更“工业化”的 main.py 样例，演示如何一步步加入：
	1.	CLI 参数解析（用 argparse）
	2.	多种仿真模式（debug vs release）
	3.	日志同时输出到屏幕和文件

你可以把它直接复制到项目根目录的 main.py，并根据需要做微调。


⸻

如何逐步扩展
	•	更多 CLI 参数
	•	加 --dt 来覆盖步长 DT
	•	加 --output-dir 指定结果图 / 数据的输出目录
	•	多种仿真模式
	•	debug：日志等级 DEBUG，并且可以在 run_simulation() 里针对 mode=="debug" 打开更详细的可视化、加速器、断点等
	•	release：默认 INFO 级别，画图时使用更简洁的样式
	•	日志文件管理
	•	使用 RotatingFileHandler 每次运行切分日志
	•	在文件名里加时间戳：log-$(date).txt
	•	配置热加载
	•	用 yaml/json 读一套更复杂的实验配置，而不仅仅是 parameters.py

这样你的 main.py 就既简洁又灵活，可随项目成长不断迭代。

'''


'''
那段更“工业化”的 main.py 其实就像大多数命令行工具一样，先用 argparse 定义了一些可选参数，然后根据你传入的参数来配置日志、覆盖仿真时长、启动仿真。下面分步给你讲明白怎么用：

⸻

1. 查看帮助信息

在终端（或 VSCode/PyCharm 的 Terminal）里，切换到项目根目录后，输入：

python main.py --help

你会看到类似这样的输出：

usage: main.py [-h] [-m {debug,release}] [-l LOG_FILE] [-t DURATION]

Airship Simulation

optional arguments:
  -h, --help            show this help message and exit
  -m {debug,release}, --mode {debug,release}
                        仿真模式：debug 会打印更多日志，release 只打印 INFO+
  -l LOG_FILE, --log-file LOG_FILE
                        如果提供，日志也会写到这个文件
  -t DURATION, --duration DURATION
                        可选：覆盖配置里的仿真总时长 T_SPAN（单位 s）

这就是所有可用参数的说明。

⸻

2. 默认运行

如果你直接敲：

python main.py

	•	使用默认模式 release（只打印 INFO 和更高级别的日志到屏幕）
	•	不写入日志文件
	•	不覆盖脚本里 config.parameters.T_SPAN（仿真时长用你在 parameters.py 里写的值）

⸻

3. 使用 debug 模式

在开发或调试时，你可能想看更详细的日志（DEBUG 级别），就加上 -m debug：

python main.py -m debug

这样会把日志等级调到 DEBUG，屏幕上会输出非常详细的内部信息（方便排错）。

⸻

4. 同时输出到日志文件

如果你想把日志也写到文件里，带上 -l 参数：

python main.py -l simulation.log

这条命令会把所有日志（INFO 及以上）同时打印到屏幕和 simulation.log 文件。
你也可以跟 debug 模式一起用：

python main.py -m debug -l debug.log



⸻

5. 临时覆盖仿真时长

假设你在 config/parameters.py 里默认 T_SPAN = 200 秒，但这次想只跑 50 秒，就加上 -t 50：

python main.py -t 50

或者连同日志文件一起：

python main.py -t 50 -l short_run.log

脚本运行时会在日志里打印：

INFO  覆盖仿真总时长：200 -> 50



⸻

小结
	•	python main.py：默认运行
	•	-m/--mode：切换 release（默认）或 debug
	•	-l/--log-file：指定一个文件路径，将日志写入该文件
	•	-t/--duration：覆盖 parameters.py 中的 T_SPAN，以秒为单位

你可以根据需要自由组合这些选项，或者用 python main.py --help 随时查看。这样就能够“调”得动你的 main.py 了。祝仿真愉快！


'''

In [ ]:
# main_2.py

import argparse
import logging
import sys
import os
import json
from datetime import datetime
from logging.handlers import RotatingFileHandler

# 可选支持 yaml
try:
    import yaml
except ImportError:
    yaml = None

from simulation.run_simulation import run_simulation


def parse_args():
    p = argparse.ArgumentParser(description="Airship Simulation")
    p.add_argument(
        "-m", "--mode",
        choices=["debug", "release"],
        default="release",
        help="仿真模式：debug 会打印 DEBUG 日志，release 只打印 INFO+"
    )
    p.add_argument(
        "-l", "--log-file",
        type=str,
        default=None,
        help="日志写入文件，如果不指定，则只打印到控制台"
    )
    p.add_argument(
        "-c", "--config",
        type=str,
        default=None,
        help="可选：外部配置文件路径（.yaml 或 .json），用来覆盖 parameters.py 中的默认常量"
    )
    p.add_argument(
        "--dt",
        type=float,
        default=None,
        help="可选：覆盖仿真步长 DT（单位：秒）"
    )
    p.add_argument(
        "--output-dir",
        type=str,
        default=None,
        help="可选：仿真结果（图表/数据）的输出目录"
    )
    return p.parse_args()


def setup_logger(mode: str, log_file: str = None):
    """配置 root logger：console + （可选）文件输出 & RotatingFileHandler"""
    level = logging.DEBUG if mode == "debug" else logging.INFO

    # Console handler
    handlers = [logging.StreamHandler(sys.stdout)]

    # 如果指定了文件输出，就加一个滚动切分的 handler
    if log_file:
        # 在文件名里自动加上时间戳
        base, ext = os.path.splitext(log_file)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        fname = f"{base}_{timestamp}{ext or '.log'}"
        fh = RotatingFileHandler(
            fname,
            maxBytes=5 * 1024 * 1024,  # 5 MB 每个日志文件
            backupCount=3,             # 保留 3 份历史
            encoding="utf-8"
        )
        handlers.append(fh)

    logging.basicConfig(
        level=level,
        format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
        handlers=handlers
    )
    logger = logging.getLogger("main")
    logger.debug(f"Logger initialized in {mode.upper()} mode")
    if log_file:
        logger.info(f"Logging to file: {fh.baseFilename}")
    return logger


def apply_external_config(path: str, logger: logging.Logger):
    """读取 yaml/json 配置，动态写入到 config.parameters 模块"""
    if not os.path.isfile(path):
        logger.error(f"配置文件不存在：{path}")
        return

    ext = os.path.splitext(path)[1].lower()
    with open(path, "r", encoding="utf-8") as f:
        if ext in (".yaml", ".yml"):
            if yaml is None:
                logger.error("要使用 YAML 配置，需要先安装 pyyaml：pip install pyyaml")
                return
            cfg = yaml.safe_load(f)
        elif ext == ".json":
            cfg = json.load(f)
        else:
            logger.error("只支持 .yaml/.yml/.json 格式的配置文件")
            return

    import config.parameters as params
    for key, val in cfg.items():
        if hasattr(params, key):
            setattr(params, key, val)
            logger.info(f"参数覆盖: {key} = {val!r}")
        else:
            logger.warning(f"未知参数跳过: {key}")

    logger.info(f"外部配置 {path} 已加载")


def main():
    args = parse_args()

    # 1. 日志初始化
    logger = setup_logger(args.mode, args.log_file)
    logger.info("=== 程序启动 ===")

    # 2. 外部配置覆盖
    if args.config:
        apply_external_config(args.config, logger)

    # 3. 覆盖单个常量：DT
    if args.dt is not None:
        import config.parameters as params
        old = params.DT
        params.DT = args.dt
        logger.info(f"覆盖 DT: {old} -> {params.DT}")

    # 4. 创建输出目录（并写入 parameters）
    if args.output_dir:
        os.makedirs(args.output_dir, exist_ok=True)
        import config.parameters as params
        params.OUTPUT_DIR = args.output_dir  # run_simulation 里可读取
        logger.info(f"输出目录: {args.output_dir}")

    # 5. 启动仿真
    run_simulation()

    logger.info("=== 仿真结束 ===")


if __name__ == "__main__":
    main()



# 然后你这样在终端跑：
# python main.py --config settings.yaml



'''
下面是一个增强版的 main.py，演示如何一步到位实现：
	•	更多 CLI 参数：--dt、--output-dir、--config
	•	多种仿真模式：debug vs release
	•	日志同时输出到屏幕、文件，并使用滚动切分（RotatingFileHandler）
	•	热加载外部 yaml 或 json 配置，动态覆盖 config.parameters 中的常量

把它放到项目根目录（和 airship/、config/、simulation/ 同级），并确保安装了 pyyaml（若要读 .yaml）：



⸻

功能逐条解读
	1.	argparse 定义 CLI 接口
	•	--mode：切换 debug（DEBUG 级别）或 release（INFO 级别）。
	•	--log-file：日志写入文件，如果提供，则自动附加时间戳并滚动切分。
	•	--config：指定一个 .yaml 或 .json，用来一次性覆盖 config/parameters.py 里的默认值。
	•	--dt：只想临时改步长，就用这个。
	•	--output-dir：仿真生成的图、数据要存哪。
	2.	日志配置
	•	StreamHandler：把日志打到终端。
	•	RotatingFileHandler：如果指定了 --log-file，就把日志写到时间戳文件里，自动切分（5 MB/文件，最多保留 3 个历史）。
	3.	热加载外部配置
	•	根据文件后缀决定走 yaml.safe_load（需安装 PyYAML）还是 json.load。
	•	逐条覆盖 config.parameters 模块中已知的属性（其余跳过，并发出警告）。
	4.	单参数覆盖
	•	如果只想改步长，用 --dt，直接给 params.DT 赋新值。
	5.	输出目录
	•	自动创建目标文件夹并写入 params.OUTPUT_DIR，你的 run_simulation() 里可以读它，把图 plt.savefig(os.path.join(OUTPUT_DIR, ...))。
	6.	统一入口
	•	run_simulation() 本身不带参数，从模块里直接读 config.parameters。
	•	if __name__=="__main__" 确保：
	•	直接 python main.py 会跑仿真；
	•	导入 这个模块时（比如未来做单元测试），不会立刻执行仿真。

这样，你的 main.py 就灵活、可配置且“工业化”了：
	•	日常调试只要 python main.py -m debug
	•	生产环境可 python main.py -c my_settings.yaml --output-dir results/ -l app.log
	•	参数维护依旧在 config/parameters.py 或外部 config 文件里搞定。

'''



'''
三、注意事项
	1.	属性名要一一对应
	•	外部文件中的顶层键（key）要和 parameters.py 里那些常量名完全一致（区分大小写）。
	•	否则会被跳过并给你一个 warning。
	2.	数据类型要对
	•	YAML/JSON 里读出来的数字、列表、字典要和 parameters.py 对应属性原来的类型兼容。
	•	例如你在 parameters.py 定义 DT = 0.05（float），就别在 YAML 里写成字符串 "0.1"，而要写成 DT: 0.1（不带引号）。
	3.	导入时机
	•	一定要在“真正用参数”之前调用 apply_external_config()，否则仿真里拿到的还是老值。
	•	推荐在 main() 一开始、run_simulation() 之前完成覆盖。
	4.	YAML 支持可选
	•	如果项目没装 pyyaml，也能读 .json。装了再读 .yaml。
	•	安装：pip install pyyaml。
	5.	不要在模块顶层改
	•	一定首先 import config.parameters，然后在函数里用 setattr。这样只有当你主动调用 apply_external_config() 时才会改，不会影响到你直接在 REPL 或者单元测试时无意中改动。


'''

In [ ]:
""" 这个是model.py 底部做的笔记
'''  
# --- 可选的测试代码块在底部 ---
# 对于 airship 包下的这些模块文件，核心的类和函数定义必须放在顶层，以便能够被其他模块导入和使用。
# 可以选择性地在这个文件底部添加 if __name__ == "__main__": 块来包含仅仅用于独立测试该模块的代码。
# 绝对不要把主要的类和函数定义放到这个 if __name__ == "__main__": 块里面。
# 这允许您通过 python airship/model.py 这样的命令来单独测试 model.py 里的 Airship 类（如果需要的话），
# 但当它被 simulation 模块导入时，测试代码不会运行。

if __name__ == "__main__":
    # 这个块只在直接运行 python airship/model.py 时执行
    print("--- 测试 Airship 类 ---")
    # 创建一个初始状态 (可能需要从 config.parameters 导入)
    initial_X = np.zeros(12)
    initial_X[6] = 10 # 设置初始速度 u=10

    # 实例化 Airship
    test_ship = Airship(initial_X)
    print("Airship 对象已创建。")

    # 测试 rhs 方法 (提供假的 tau, disturbance, wind)
    current_t = 0.0
    fake_tau = np.zeros(6)
    def fake_dist(t): return np.zeros(6)
    fake_wind = np.array([1.0, 0, 0])

    try:
        dXdt = test_ship.rhs(current_t, test_ship.get_state(), fake_tau, fake_dist, fake_wind)
        print(f"rhs 方法在 t={current_t} 时计算得到的 dXdt:\n{dXdt}")
        assert dXdt.shape == (12,) # 检查输出形状
        print("rhs 方法基本运行测试通过。")
    except Exception as e:
        print(f"测试 rhs 时出错: {e}")

    # 可以添加更多针对特定方法的测试



'''


"""



In [ ]:
        #--- 计算总推力 (Calculate Total Thrust and  Torque) ---
        # --- 计算推力和推力力矩 (Calculate Thrust Forces and Moments) ---
        # 从tau中提取推力参数 / Extract thrust parameters from tau
        T_mag = tau[0]  # 推力大小 / Thrust magnitude
        mu = tau[1]     # 水平面内的推力偏转角 / Thrust deflection angle in the horizontal plane
        nu = tau[2]     # 垂直面内的推力偏转角 / Thrust deflection angle in the vertical plane

        # 计算右侧推力向量 / Calculate right thrust vector
        T_vec_r = np.array([
            [T_mag * np.cos(mu) * np.cos(nu)],
            [T_mag * np.sin(mu)],
            [T_mag * np.cos(mu) * np.sin(nu)]
        ])

        # 计算左侧推力向量 / Calculate left thrust vector
        T_vec_l = np.array([
            [T_mag * np.cos(mu) * np.cos(nu)],
            [T_mag * np.sin(mu)],
            [T_mag * np.cos(mu) * np.sin(nu)]
        ])

        # 计算总推力 /  Calculate total thrust
        T_total = T_vec_r + T_vec_l

        # 计算推力力矩 / Calculate thrust torque
        rp_r_1d = self.rp_r_vec.flatten()
        rp_l_1d = self.rp_l_vec.flatten()
        tau_r = np.cross(rp_r_1d, T_vec_r.flatten()).reshape(3, 1)
        tau_l = np.cross(rp_l_1d, T_vec_l.flatten()).reshape(3, 1)
        tau_vec = tau_r + tau_l       